
# Fleet Reliability Data Engineering Platform
# Objective:
Clean and validate synthetic fleet reliability datasets.

This notebook cleans and validates the raw synthetic datasets created in `01_data_generation.ipynb`.

## Tasks performed
- Load raw CSV datasets
- Check structure, data types, and null values
- Remove duplicates
- Convert date fields
- Standardize text values
- Handle missing values
- Validate numeric ranges
- Enforce referential integrity
- Save cleaned datasets to `data/processed/`



In [1]:
# import libraries
import pandas as pd
import numpy as np
import os

In [2]:
# Setting display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [3]:
# Define File Paths
raw_path = "data/raw"
processed_path = "data/processed"

os.makedirs(processed_path, exist_ok=True)

In [6]:
from google.colab import files
uploaded = files.upload()

Saving warranty_claims.csv to warranty_claims.csv
Saving vehicles.csv to vehicles (1).csv
Saving telemetry_summary.csv to telemetry_summary.csv
Saving service_records.csv to service_records.csv
Saving parts_failure.csv to parts_failure.csv


In [7]:
# load Datasets

vehicles = pd.read_csv("vehicles.csv")
service_records = pd.read_csv("service_records.csv")
telemetry_summary = pd.read_csv("telemetry_summary.csv")
warranty_claims = pd.read_csv("warranty_claims.csv")
parts_failure = pd.read_csv("parts_failure.csv")

In [8]:
# Quick Shape Check

print("vehicles:", vehicles.shape)
print("service_records:", service_records.shape)
print("telemetry_summary:", telemetry_summary.shape)
print("warranty_claims:", warranty_claims.shape)
print("parts_failure:", parts_failure.shape)

vehicles: (50000, 9)
service_records: (300000, 10)
telemetry_summary: (500000, 9)
warranty_claims: (100000, 6)
parts_failure: (120000, 6)


In [9]:
# Preview Data

print("vehicles:", vehicles.shape)
print("service_records:", service_records.shape)
print("telemetry_summary:", telemetry_summary.shape)
print("warranty_claims:", warranty_claims.shape)
print("parts_failure:", parts_failure.shape)

vehicles: (50000, 9)
service_records: (300000, 10)
telemetry_summary: (500000, 9)
warranty_claims: (100000, 6)
parts_failure: (120000, 6)


In [10]:
#Check data types
print("Vehicles dtypes")
print(vehicles.dtypes)
print()

print("Service Records dtypes")
print(service_records.dtypes)
print()

print("Telemetry Summary dtypes")
print(telemetry_summary.dtypes)
print()

print("Warranty Claims dtypes")
print(warranty_claims.dtypes)
print()

print("Parts Failure dtypes")
print(parts_failure.dtypes)

Vehicles dtypes
vehicle_id        int64
vin              object
model            object
model_year        int64
battery_type     object
delivery_date    object
region           object
mileage           int64
fleet_type       object
dtype: object

Service Records dtypes
service_id             int64
vehicle_id             int64
service_date          object
service_center        object
issue_category        object
issue_subcategory     object
severity              object
repair_cost          float64
downtime_days        float64
repeat_issue_flag      int64
dtype: object

Telemetry Summary dtypes
telemetry_id              int64
vehicle_id                int64
date                     object
battery_health          float64
avg_motor_temp          float64
avg_brake_temp          float64
tire_pressure_alerts      int64
software_version         object
warning_count             int64
dtype: object

Warranty Claims dtypes
claim_id          int64
vehicle_id        int64
claim_date       object
co

In [11]:
# Check missing values

print("Missing values in vehicles")
print(vehicles.isnull().sum())
print()

print("Missing values in service_records")
print(service_records.isnull().sum())
print()

print("Missing values in telemetry_summary")
print(telemetry_summary.isnull().sum())
print()

print("Missing values in warranty_claims")
print(warranty_claims.isnull().sum())
print()

print("Missing values in parts_failure")
print(parts_failure.isnull().sum())

Missing values in vehicles
vehicle_id       0
vin              0
model            0
model_year       0
battery_type     0
delivery_date    0
region           0
mileage          0
fleet_type       0
dtype: int64

Missing values in service_records
service_id           0
vehicle_id           0
service_date         0
service_center       0
issue_category       0
issue_subcategory    0
severity             0
repair_cost          0
downtime_days        0
repeat_issue_flag    0
dtype: int64

Missing values in telemetry_summary
telemetry_id            0
vehicle_id              0
date                    0
battery_health          0
avg_motor_temp          0
avg_brake_temp          0
tire_pressure_alerts    0
software_version        0
warning_count           0
dtype: int64

Missing values in warranty_claims
claim_id        0
vehicle_id      0
claim_date      0
component       0
claim_amount    0
claim_status    0
dtype: int64

Missing values in parts_failure
failure_id       0
vehicle_id       0


In [12]:
# Check Duplicates

print("Duplicate rows")
print("vehicles:", vehicles.duplicated().sum())
print("service_records:", service_records.duplicated().sum())
print("telemetry_summary:", telemetry_summary.duplicated().sum())
print("warranty_claims:", warranty_claims.duplicated().sum())
print("parts_failure:", parts_failure.duplicated().sum())

Duplicate rows
vehicles: 0
service_records: 0
telemetry_summary: 0
warranty_claims: 0
parts_failure: 0


In [13]:
# Convert Date Columns

vehicles["delivery_date"] = pd.to_datetime(vehicles["delivery_date"], errors="coerce")
service_records["service_date"] = pd.to_datetime(service_records["service_date"], errors="coerce")
telemetry_summary["date"] = pd.to_datetime(telemetry_summary["date"], errors="coerce")
warranty_claims["claim_date"] = pd.to_datetime(warranty_claims["claim_date"], errors="coerce")
parts_failure["failure_date"] = pd.to_datetime(parts_failure["failure_date"], errors="coerce")

In [14]:
# Check Date conversion issues

print("Invalid date counts after conversion:")
print("vehicles.delivery_date:", vehicles["delivery_date"].isnull().sum())
print("service_records.service_date:", service_records["service_date"].isnull().sum())
print("telemetry_summary.date:", telemetry_summary["date"].isnull().sum())
print("warranty_claims.claim_date:", warranty_claims["claim_date"].isnull().sum())
print("parts_failure.failure_date:", parts_failure["failure_date"].isnull().sum())

Invalid date counts after conversion:
vehicles.delivery_date: 0
service_records.service_date: 0
telemetry_summary.date: 0
warranty_claims.claim_date: 0
parts_failure.failure_date: 0


In [15]:
# Drop rows with invalid critical dates

vehicles = vehicles.dropna(subset=["delivery_date"])
service_records = service_records.dropna(subset=["service_date"])
telemetry_summary = telemetry_summary.dropna(subset=["date"])
warranty_claims = warranty_claims.dropna(subset=["claim_date"])
parts_failure = parts_failure.dropna(subset=["failure_date"])

print("Rows with invalid critical dates removed.")

Rows with invalid critical dates removed.


In [16]:
# Standardize Text Columns Helper

def clean_text_column(df, col):
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)
    return df

In [17]:
# Clean text columns

vehicle_text_cols = ["vin", "model", "battery_type", "region", "fleet_type"]
service_text_cols = ["service_center", "issue_category", "issue_subcategory", "severity"]
telemetry_text_cols = ["software_version"]
warranty_text_cols = ["component", "claim_status"]
parts_failure_text_cols = ["component", "failure_type"]

for col in vehicle_text_cols:
    vehicles = clean_text_column(vehicles, col)

for col in service_text_cols:
    service_records = clean_text_column(service_records, col)

for col in telemetry_text_cols:
    telemetry_summary = clean_text_column(telemetry_summary, col)

for col in warranty_text_cols:
    warranty_claims = clean_text_column(warranty_claims, col)

for col in parts_failure_text_cols:
    parts_failure = clean_text_column(parts_failure, col)

print("Text columns standardized.")

Text columns standardized.


In [18]:
# Standardize category capitalization

vehicles["model"] = vehicles["model"].str.title()
vehicles["battery_type"] = vehicles["battery_type"].str.upper()
vehicles["region"] = vehicles["region"].str.title()
vehicles["fleet_type"] = vehicles["fleet_type"].str.title()

service_records["service_center"] = service_records["service_center"].str.title()
service_records["issue_category"] = service_records["issue_category"].str.title()
service_records["issue_subcategory"] = service_records["issue_subcategory"].str.title()
service_records["severity"] = service_records["severity"].str.title()

warranty_claims["component"] = warranty_claims["component"].str.title()
warranty_claims["claim_status"] = warranty_claims["claim_status"].str.title()

parts_failure["component"] = parts_failure["component"].str.title()
parts_failure["failure_type"] = parts_failure["failure_type"].str.title()

In [19]:
# Convert numeric columns

vehicles["vehicle_id"] = pd.to_numeric(vehicles["vehicle_id"], errors="coerce")
vehicles["model_year"] = pd.to_numeric(vehicles["model_year"], errors="coerce")
vehicles["mileage"] = pd.to_numeric(vehicles["mileage"], errors="coerce")

service_records["service_id"] = pd.to_numeric(service_records["service_id"], errors="coerce")
service_records["vehicle_id"] = pd.to_numeric(service_records["vehicle_id"], errors="coerce")
service_records["repair_cost"] = pd.to_numeric(service_records["repair_cost"], errors="coerce")
service_records["downtime_days"] = pd.to_numeric(service_records["downtime_days"], errors="coerce")
service_records["repeat_issue_flag"] = pd.to_numeric(service_records["repeat_issue_flag"], errors="coerce")

telemetry_summary["telemetry_id"] = pd.to_numeric(telemetry_summary["telemetry_id"], errors="coerce")
telemetry_summary["vehicle_id"] = pd.to_numeric(telemetry_summary["vehicle_id"], errors="coerce")
telemetry_summary["battery_health"] = pd.to_numeric(telemetry_summary["battery_health"], errors="coerce")
telemetry_summary["avg_motor_temp"] = pd.to_numeric(telemetry_summary["avg_motor_temp"], errors="coerce")
telemetry_summary["avg_brake_temp"] = pd.to_numeric(telemetry_summary["avg_brake_temp"], errors="coerce")
telemetry_summary["tire_pressure_alerts"] = pd.to_numeric(telemetry_summary["tire_pressure_alerts"], errors="coerce")
telemetry_summary["warning_count"] = pd.to_numeric(telemetry_summary["warning_count"], errors="coerce")

warranty_claims["claim_id"] = pd.to_numeric(warranty_claims["claim_id"], errors="coerce")
warranty_claims["vehicle_id"] = pd.to_numeric(warranty_claims["vehicle_id"], errors="coerce")
warranty_claims["claim_amount"] = pd.to_numeric(warranty_claims["claim_amount"], errors="coerce")

parts_failure["failure_id"] = pd.to_numeric(parts_failure["failure_id"], errors="coerce")
parts_failure["vehicle_id"] = pd.to_numeric(parts_failure["vehicle_id"], errors="coerce")
parts_failure["replaced_flag"] = pd.to_numeric(parts_failure["replaced_flag"], errors="coerce")

In [20]:
# Drop rows with missing primary identifiers

vehicles = vehicles.dropna(subset=["vehicle_id"])
service_records = service_records.dropna(subset=["service_id", "vehicle_id"])
telemetry_summary = telemetry_summary.dropna(subset=["telemetry_id", "vehicle_id"])
warranty_claims = warranty_claims.dropna(subset=["claim_id", "vehicle_id"])
parts_failure = parts_failure.dropna(subset=["failure_id", "vehicle_id"])

print("Rows with missing primary identifiers removed.")

Rows with missing primary identifiers removed.


In [21]:
# Convert IDs to integer

vehicles["vehicle_id"] = vehicles["vehicle_id"].astype(int)
vehicles["model_year"] = vehicles["model_year"].astype(int)

service_records["service_id"] = service_records["service_id"].astype(int)
service_records["vehicle_id"] = service_records["vehicle_id"].astype(int)
service_records["repeat_issue_flag"] = service_records["repeat_issue_flag"].fillna(0).astype(int)

telemetry_summary["telemetry_id"] = telemetry_summary["telemetry_id"].astype(int)
telemetry_summary["vehicle_id"] = telemetry_summary["vehicle_id"].astype(int)
telemetry_summary["tire_pressure_alerts"] = telemetry_summary["tire_pressure_alerts"].fillna(0).astype(int)
telemetry_summary["warning_count"] = telemetry_summary["warning_count"].fillna(0).astype(int)

warranty_claims["claim_id"] = warranty_claims["claim_id"].astype(int)
warranty_claims["vehicle_id"] = warranty_claims["vehicle_id"].astype(int)

parts_failure["failure_id"] = parts_failure["failure_id"].astype(int)
parts_failure["vehicle_id"] = parts_failure["vehicle_id"].astype(int)
parts_failure["replaced_flag"] = parts_failure["replaced_flag"].fillna(0).astype(int)

In [22]:
# Handle missing numeric values

service_records["repair_cost"] = service_records["repair_cost"].fillna(service_records["repair_cost"].median())
service_records["downtime_days"] = service_records["downtime_days"].fillna(service_records["downtime_days"].median())

telemetry_summary["battery_health"] = telemetry_summary["battery_health"].fillna(telemetry_summary["battery_health"].median())
telemetry_summary["avg_motor_temp"] = telemetry_summary["avg_motor_temp"].fillna(telemetry_summary["avg_motor_temp"].median())
telemetry_summary["avg_brake_temp"] = telemetry_summary["avg_brake_temp"].fillna(telemetry_summary["avg_brake_temp"].median())

warranty_claims["claim_amount"] = warranty_claims["claim_amount"].fillna(warranty_claims["claim_amount"].median())

In [23]:
# Handle missing categorical values
service_records["service_center"] = service_records["service_center"].replace("Nan", np.nan).fillna("Unknown")
service_records["issue_category"] = service_records["issue_category"].replace("Nan", np.nan).fillna("Unknown")
service_records["issue_subcategory"] = service_records["issue_subcategory"].replace("Nan", np.nan).fillna("Unknown")
service_records["severity"] = service_records["severity"].replace("Nan", np.nan).fillna("Unknown")

telemetry_summary["software_version"] = telemetry_summary["software_version"].replace("Nan", np.nan).fillna("Unknown")

warranty_claims["component"] = warranty_claims["component"].replace("Nan", np.nan).fillna("Unknown")
warranty_claims["claim_status"] = warranty_claims["claim_status"].replace("Nan", np.nan).fillna("Pending")

parts_failure["component"] = parts_failure["component"].replace("Nan", np.nan).fillna("Unknown")
parts_failure["failure_type"] = parts_failure["failure_type"].replace("Nan", np.nan).fillna("Unknown")


In [24]:
# Remove impossible or invalid values

vehicles = vehicles[vehicles["mileage"] >= 0]
vehicles = vehicles[(vehicles["model_year"] >= 2015) & (vehicles["model_year"] <= 2026)]

service_records = service_records[service_records["repair_cost"] >= 0]
service_records = service_records[service_records["downtime_days"] >= 0]

telemetry_summary = telemetry_summary[
    (telemetry_summary["battery_health"] >= 0) & (telemetry_summary["battery_health"] <= 100)
]
telemetry_summary = telemetry_summary[telemetry_summary["avg_motor_temp"] >= 0]
telemetry_summary = telemetry_summary[telemetry_summary["avg_brake_temp"] >= 0]
telemetry_summary = telemetry_summary[telemetry_summary["tire_pressure_alerts"] >= 0]
telemetry_summary = telemetry_summary[telemetry_summary["warning_count"] >= 0]

warranty_claims = warranty_claims[warranty_claims["claim_amount"] >= 0]

parts_failure = parts_failure[parts_failure["replaced_flag"].isin([0, 1])]

print("Invalid records removed.")

Invalid records removed.


In [25]:
# Validate allowed values

valid_models = ["Model S", "Model 3", "Model X", "Model Y", "Cybertruck"]
valid_regions = ["North America", "Europe", "Asia Pacific", "Middle East"]
valid_battery_types = ["LFP", "NCA", "NCM"]
valid_fleet_types = ["Retail", "Commercial", "Internal Test"]

valid_issue_categories = ["Battery", "Brakes", "Motor", "Suspension", "Software", "Hvac", "Electrical"]
valid_severity = ["Low", "Medium", "High", "Critical", "Unknown"]
valid_claim_status = ["Approved", "Pending", "Rejected"]

In [26]:
# Filter invalid categories

vehicles = vehicles[
    vehicles["model"].isin(valid_models) &
    vehicles["region"].isin(valid_regions) &
    vehicles["battery_type"].isin(valid_battery_types) &
    vehicles["fleet_type"].isin(valid_fleet_types)
]

service_records = service_records[
    service_records["issue_category"].isin(valid_issue_categories) &
    service_records["severity"].isin(valid_severity)
]

warranty_claims = warranty_claims[
    warranty_claims["claim_status"].isin(valid_claim_status)
]

print("Category validation completed.")

Category validation completed.


In [27]:
# Referential integrity check

valid_vehicle_ids = set(vehicles["vehicle_id"])

service_records = service_records[service_records["vehicle_id"].isin(valid_vehicle_ids)]
telemetry_summary = telemetry_summary[telemetry_summary["vehicle_id"].isin(valid_vehicle_ids)]
warranty_claims = warranty_claims[warranty_claims["vehicle_id"].isin(valid_vehicle_ids)]
parts_failure = parts_failure[parts_failure["vehicle_id"].isin(valid_vehicle_ids)]

print("Referential integrity enforced.")

Referential integrity enforced.


In [28]:
# Check for duplicate primary keys

print("Duplicate primary keys")
print("vehicles.vehicle_id:", vehicles["vehicle_id"].duplicated().sum())
print("service_records.service_id:", service_records["service_id"].duplicated().sum())
print("telemetry_summary.telemetry_id:", telemetry_summary["telemetry_id"].duplicated().sum())
print("warranty_claims.claim_id:", warranty_claims["claim_id"].duplicated().sum())
print("parts_failure.failure_id:", parts_failure["failure_id"].duplicated().sum())

Duplicate primary keys
vehicles.vehicle_id: 0
service_records.service_id: 0
telemetry_summary.telemetry_id: 0
warranty_claims.claim_id: 0
parts_failure.failure_id: 0


In [29]:
# Remove duplicate primary keys if any

vehicles = vehicles.drop_duplicates(subset=["vehicle_id"])
service_records = service_records.drop_duplicates(subset=["service_id"])
telemetry_summary = telemetry_summary.drop_duplicates(subset=["telemetry_id"])
warranty_claims = warranty_claims.drop_duplicates(subset=["claim_id"])
parts_failure = parts_failure.drop_duplicates(subset=["failure_id"])

print("Duplicate primary keys removed if present.")

Duplicate primary keys removed if present.


In [30]:
# missing values check

print("Final missing values in vehicles")
print(vehicles.isnull().sum())
print()

print("Final missing values in service_records")
print(service_records.isnull().sum())
print()

print("Final missing values in telemetry_summary")
print(telemetry_summary.isnull().sum())
print()

print("Final missing values in warranty_claims")
print(warranty_claims.isnull().sum())
print()

print("Final missing values in parts_failure")
print(parts_failure.isnull().sum())

Final missing values in vehicles
vehicle_id       0
vin              0
model            0
model_year       0
battery_type     0
delivery_date    0
region           0
mileage          0
fleet_type       0
dtype: int64

Final missing values in service_records
service_id           0
vehicle_id           0
service_date         0
service_center       0
issue_category       0
issue_subcategory    0
severity             0
repair_cost          0
downtime_days        0
repeat_issue_flag    0
dtype: int64

Final missing values in telemetry_summary
telemetry_id            0
vehicle_id              0
date                    0
battery_health          0
avg_motor_temp          0
avg_brake_temp          0
tire_pressure_alerts    0
software_version        0
warning_count           0
dtype: int64

Final missing values in warranty_claims
claim_id        0
vehicle_id      0
claim_date      0
component       0
claim_amount    0
claim_status    0
dtype: int64

Final missing values in parts_failure
failure_

In [31]:
# shape check

print("Final cleaned shapes")
print("vehicles:", vehicles.shape)
print("service_records:", service_records.shape)
print("telemetry_summary:", telemetry_summary.shape)
print("warranty_claims:", warranty_claims.shape)
print("parts_failure:", parts_failure.shape)

Final cleaned shapes
vehicles: (50000, 9)
service_records: (300000, 10)
telemetry_summary: (500000, 9)
warranty_claims: (100000, 6)
parts_failure: (120000, 6)


# Data Quality Summary

# Vehicles summary

In [33]:
print("Vehicles summary")
display(vehicles.describe(include="all"))

Vehicles summary


,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type
count,50000.000000,50000,50000,50000.000000,50000,50000,50000,50000.000000,50000
unique,NaN,50000,5,NaN,3,NaN,4,NaN,3
top,NaN,5YJ00000000049984,Model Y,NaN,NCA,NaN,North America,NaN,Retail
freq,NaN,1,17100,NaN,17626,NaN,22550,NaN,38922
mean,25000.500000,NaN,NaN,2022.340800,NaN,2022-07-01 03:01:09.120000,NaN,54839.938500,NaN
min,1.000000,NaN,NaN,2019.000000,NaN,2019-01-01 00:00:00,NaN,502.000000,NaN
25%,12500.750000,NaN,NaN,2021.000000,NaN,2020-10-04 00:00:00,NaN,21577.750000,NaN
50%,25000.500000,NaN,NaN,2023.000000,NaN,2022-06-29 00:00:00,NaN,46817.500000,NaN
75%,37500.250000,NaN,NaN,2024.000000,NaN,2024-03-30 00:00:00,NaN,81030.250000,NaN
max,50000.000000,NaN,NaN,2025.000000,NaN,2025-12-31 00:00:00,NaN,179982.000000,NaN


# Service records summary

In [34]:
print("Service records summary")
display(service_records.describe(include="all"))

Service records summary


,service_id,vehicle_id,service_date,service_center,issue_category,issue_subcategory,severity,repair_cost,downtime_days,repeat_issue_flag
count,300000.000000,300000.000000,300000,300000,300000,300000,300000,300000.000000,300000.000000,300000.000000
unique,NaN,NaN,NaN,6,7,21,4,NaN,NaN,NaN
top,NaN,NaN,NaN,Toronto,Software,Charging Failure,Low,NaN,NaN,NaN
freq,NaN,NaN,NaN,50199,53964,18121,119730,NaN,NaN,NaN
mean,150000.500000,25035.292717,2023-08-02 04:46:34.751999232,NaN,NaN,NaN,NaN,1873.330389,3.604929,0.151643
min,1.000000,1.000000,2021-01-01 00:00:00,NaN,NaN,NaN,NaN,50.020000,0.000000,0.000000
25%,75000.750000,12527.000000,2022-04-19 00:00:00,NaN,NaN,NaN,NaN,655.840000,2.500000,0.000000
50%,150000.500000,25049.500000,2023-08-02 00:00:00,NaN,NaN,NaN,NaN,1255.990000,3.600000,0.000000
75%,225000.250000,37563.000000,2024-11-16 00:00:00,NaN,NaN,NaN,NaN,2493.030000,4.700000,0.000000
max,300000.000000,50000.000000,2026-03-01 00:00:00,NaN,NaN,NaN,NaN,8518.900000,11.200000,1.000000


# Telemetry summary

In [35]:
print("Telemetry summary")
display(telemetry_summary.describe(include="all"))

Telemetry summary


,telemetry_id,vehicle_id,date,battery_health,avg_motor_temp,avg_brake_temp,tire_pressure_alerts,software_version,warning_count
count,500000.000000,500000.000000,500000,500000.000000,500000.000000,500000.000000,500000.000000,500000,500000.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.44.3,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100606,NaN
mean,250000.500000,25001.275202,2025-08-01 02:45:53.971199744,91.515047,72.006214,58.422328,0.600372,NaN,2.180530
min,1.000000,1.000000,2025-01-01 00:00:00,70.140000,24.620000,17.490000,0.000000,NaN,0.000000
25%,125000.750000,12507.750000,2025-04-17 00:00:00,87.640000,65.270000,52.290000,0.000000,NaN,1.000000
50%,250000.500000,25006.000000,2025-08-01 00:00:00,92.180000,72.000000,58.429855,0.000000,NaN,2.000000
75%,375000.250000,37523.250000,2025-11-15 00:00:00,95.850000,78.740000,64.550000,1.000000,NaN,3.000000
max,500000.000000,50000.000000,2026-03-01 00:00:00,100.000000,120.900000,100.053988,7.000000,NaN,14.000000


# Warranty claims summary

In [36]:
print("Warranty claims summary")
display(warranty_claims.describe(include="all"))

Warranty claims summary


,claim_id,vehicle_id,claim_date,component,claim_amount,claim_status
count,100000.000000,100000.000000,100000,100000,100000.000000,100000
unique,NaN,NaN,NaN,21,NaN,3
top,NaN,NaN,NaN,Battery Pack,NaN,Approved
freq,NaN,NaN,NaN,6701,NaN,67973
mean,50000.500000,25060.713420,2023-07-31 04:34:20.928000,NaN,2557.705428,NaN
min,1.000000,1.000000,2021-01-01 00:00:00,NaN,51.220000,NaN
25%,25000.750000,12570.750000,2022-04-15 00:00:00,NaN,836.707500,NaN
50%,50000.500000,25113.000000,2023-08-01 00:00:00,NaN,1702.385000,NaN
75%,75000.250000,37525.000000,2024-11-13 00:00:00,NaN,3294.625000,NaN
max,100000.000000,50000.000000,2026-03-01 00:00:00,NaN,11699.870000,NaN


# Parts failure summary

In [37]:
print("Parts failure summary")
display(parts_failure.describe(include="all"))

Parts failure summary


,failure_id,vehicle_id,component,failure_date,failure_type,replaced_flag
count,120000.000000,120000.000000,120000,120000,120000,120000.000000
unique,NaN,NaN,21,NaN,5,NaN
top,NaN,NaN,Battery Pack,NaN,Software Error,NaN
freq,NaN,NaN,8010,NaN,34564,NaN
mean,60000.500000,25012.773958,NaN,2023-07-30 05:28:14.160000,NaN,0.750017
min,1.000000,1.000000,NaN,2021-01-01 00:00:00,NaN,0.000000
25%,30000.750000,12532.000000,NaN,2022-04-15 00:00:00,NaN,1.000000
50%,60000.500000,25075.500000,NaN,2023-07-28 00:00:00,NaN,1.000000
75%,90000.250000,37428.250000,NaN,2024-11-13 00:00:00,NaN,1.000000
max,120000.000000,50000.000000,NaN,2026-03-01 00:00:00,NaN,1.000000




---



In [38]:
# Save Cleaned Datasets

vehicles.to_csv(f"{processed_path}/vehicles_cleaned.csv", index=False)
service_records.to_csv(f"{processed_path}/service_records_cleaned.csv", index=False)
telemetry_summary.to_csv(f"{processed_path}/telemetry_summary_cleaned.csv", index=False)
warranty_claims.to_csv(f"{processed_path}/warranty_claims_cleaned.csv", index=False)
parts_failure.to_csv(f"{processed_path}/parts_failure_cleaned.csv", index=False)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [41]:

print("Processed files are available in:", processed_path)

Processed files are available in: data/processed
